In [152]:
from reload import reload_package
reload_package(namespace=globals())

Reloaded 9 modules, rebound 9 names.


In [153]:
import altair as alt
import numpy as np
import polars as pl
import torch

from ts_uncertainty.forecasting import ForecastModel
from ts_uncertainty.metrics import mean_absolute_scaled_error, scaled_quantile_loss

alt.renderers.enable("jupyter", offline=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data

In [154]:
from ts_uncertainty.chaotic import ChaoticGenerator
from ts_uncertainty.injection import Injector
from ts_uncertainty.plotting import _style, plot_sample
from ts_uncertainty.synthetic import SyntheticGenerator
from ts_uncertainty.real import RealSampler

# data settings
context_length = 256
horizon_length = 128
samples_batch = 100
seed = 42

## Synthetic Samples

In [5]:
sin_gen = SyntheticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    seasonal=True,
    covariate=False,
    periods=(24,),             # period of the sine
    # n_periods=1,
    n_harmonics=3,             # 1 harmonic = pure sine, no extra shape
    # level_range=(0.0, 0.0),    # centered at 0
    slope_range=(0.0, 0.0),    # no trend
    # amplitude_range=(1.0, 1.0),
    seed=seed
)

# generate samples
samples_synthetic = sin_gen.sample_batch(samples_batch)

In [6]:
sample_to_plot = 0
plot_sample(samples_synthetic[sample_to_plot], title=f"Synthetic Sample {sample_to_plot}")

## Chaotic Samples

In [8]:
chaos_gen = ChaoticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    systems=None,                    # None = all ~120 default systems
    dim=0,                           # observed coordinate, None = random per sample
    pts_per_period_range=(20, 30),   # time steps per dominant period (smaller = more irregular)
    level_range=(0.0, 0.0),          # centered at 0
    slope_range=(0.0, 0.0),          # no trend
    amplitude_range=(1.0, 1.0),      # z-normalised signal, std ≈ 1
    seed=seed,
)

# generate samples
samples_chaotic = chaos_gen.sample_batch(samples_batch)

In [9]:
sample_to_plot = 0
plot_sample(samples_chaotic[sample_to_plot], title=f"Chaotic Sample {sample_to_plot}")

## Real Sample
Download the data from the notebook [data_download.ipynb](data_download.ipynb) before.

In [10]:
data_path_test = "../data/test.csv"
data_test = pl.read_csv(data_path_test, try_parse_dates=True)

number_series = len(data_test['series_index'].unique())
print(f"Number of time series in the test set: {number_series}")
data_test.head()

Number of time series in the test set: 361


timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2025-11-19 10:00:00,0.36,"""home_electricity""",0
2025-11-19 10:15:00,0.124,"""home_electricity""",0
2025-11-19 10:30:00,0.108,"""home_electricity""",0
2025-11-19 10:45:00,0.088,"""home_electricity""",0
2025-11-19 11:00:00,0.116,"""home_electricity""",0


In [11]:
real_gen = RealSampler(
    df=data_test,
    context_length=context_length,
    horizon=horizon_length,
    normalize='none', # dont normalize
    seed=seed,
)

# generate samples
samples_real = real_gen.sample_batch(samples_batch)

In [12]:
sample_to_plot = 0
plot_sample(samples_real[sample_to_plot], title=f"Real Sample {sample_to_plot}")

# Models

In [143]:
models = {
    "AutoTheta": {"name": "autotheta"},
    "Chronos-2": {'name': 'chronos2'},
    "TiRex-2": {'name': 'tirex2'},
    "Timestral (123M)": {'name': 'timestral', 'size': '123M'},
}

# prepare and download all models
for key, m in models.items():
    print(f"Preparing model: {key}")
    model = ForecastModel(**m)

Preparing model: AutoTheta
Preparing model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Preparing model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Preparing model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

# Experiments

In [144]:
def evaluate_models_on_samples(models, samples):
    results = []
    scores_mase = []
    scores_sql = []

    for key, m in models.items():
        print(f"Evaluating model: {key}")
        model = ForecastModel(**m)
        fc = model.forecast(samples)

        fc_m = fc.metric_inputs()
        score_mase = mean_absolute_scaled_error(fc_m["y_true"], fc_m["y_pred"], fc_m["y_train"], fc_m["season_length"])
        score_sql = scaled_quantile_loss(fc_m["y_true"], fc_m["y_quantiles"], fc_m["y_train"], fc_m["quantiles"], fc_m["season_length"])

        results.append(fc)
        scores_mase.append(score_mase.mean())
        scores_sql.append(score_sql.mean())

    scores = pl.DataFrame({
        "model": list(models.keys()),
        "mase": [float(x.squeeze()) for x in scores_mase],
        "sql": [float(x.squeeze()) for x in scores_sql],
    })
    return results, scores

In [145]:
batch_size = 10  # define the batch size for evaluation
data_samples = {
    "synthetic": samples_synthetic[:batch_size],
    "chaotic": samples_chaotic[:batch_size],
    "real": samples_real[:batch_size],
}

## No Injection

In [146]:
fc_results = {}
fc_scores = {}
for type_s, s in data_samples.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    print(f"Scores {type_s} samples:")
    print(scores)
    # save results and scores for a dataset
    fc_results[type_s] = results
    fc_scores[type_s] = scores

Forecasting synthetic samples..., number of samples: 10
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


... (15 outputs left out)

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Scores real samples:
shape: (4, 3)
┌──────────────────┬──────────┬──────────┐
│ model            ┆ mase     ┆ sql      │
│ ---              ┆ ---      ┆ ---      │
│ str              ┆ f64      ┆ f64      │
╞══════════════════╪══════════╪══════════╡
│ AutoTheta        ┆ 1.264568 ┆ 1.089648 │
│ Chronos-2        ┆ 0.769237 ┆ 0.624378 │
│ TiRex-2          ┆ 0.805082 ┆ 0.633168 │
│ Timestral (123M) ┆ 0.768344 ┆ 0.615806 │
└──────────────────┴──────────┴──────────┘


In [157]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

s = data_samples[dataset_to_plot]
results = fc_results[dataset_to_plot]
smp = s[sample_to_plot]
# y-range from the data and all models except the first (AutoTheta): its wide bands would squash the plot
vals = [smp.context, smp.target] + [np.asarray(r.quantiles[sample_to_plot]).ravel() for r in results[1:]]
lo, hi = float(min(v.min() for v in vals)), float(max(v.max() for v in vals))
pad = 0.05 * (hi - lo)
y_domain = (lo - pad, hi + pad)

charts = []
for i, key in enumerate(models.keys()):
    c = plot_sample(smp, prediction=results[i].quantiles[sample_to_plot], quantiles=results[i].quantile_levels,
                    title=key, width=1000, height=200, y_domain=y_domain)
    c.config = alt.Undefined
    charts.append(c)
grid_chart = _style(alt.concat(*charts, columns=1, spacing=24))
grid_chart

## Noise

In [ ]:
samples_synthetic



inj = Injector(
    noise=False,
    shift=True,
    shift_covariate=True,
    shift_recurring_prob=0.8,
    shift_duration_range=(0.1, 0.25),
    seed=seed,
)
s = inj(sin_gen.sample())
plot_sample(s, title="Sin Sample")